# 🛡️ Project Panopticon: Intelligent Exam Proctoring AI
**EduGuard AI — Machine Learning Division**

**Intern:** Varun Kumar  
**Project:** Project Panopticon: Intelligent Exam Proctoring

## Executive Summary
This notebook builds the required time-series classification pipeline for detecting suspicious exam behavior while reducing false accusations. It follows the company problem statement: asynchronous timestamp alignment, missing-sensor handling, rolling-window feature engineering, class-balanced classification, probability-based prediction, and a strict 90% decision threshold.

## Roadmap
1. Data ingestion
2. Asynchronous temporal alignment with `merge_asof`
3. Missing-value treatment
4. 10-second rolling features
5. Class-balanced Random Forest
6. Probability-based decision thresholds
7. Precision-Recall evaluation
8. Executive recommendation

In [ ]:
# ==========================================
# 0. IMPORT LIBRARIES
# ==========================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score
)

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 6)

print("✅ Libraries imported successfully.")

## Phase 1 — Data Ingestion & Temporal Alignment

The company identifies an asynchronous merge problem: video telemetry is recorded every second, while system events occur only when an event happens. Therefore, a normal `pd.merge()` is not appropriate. We use `pd.merge_asof()` with backward matching so each video timestamp receives the most recent system event. 

In [ ]:
# ==========================================
# 1A. LOCATE THE PROVIDED CSV FILES
# ==========================================
# The notebook supports both the original company filenames and
# the filenames used when the files were downloaded/uploaded here.

candidate_video = [
    "video_telemetry.csv",
    "video_telemetry (1) (1).csv",
]
candidate_system = [
    "system_events.csv",
    "system_events (1) (1).csv",
]

video_path = next((p for p in candidate_video if os.path.exists(p)), None)
system_path = next((p for p in candidate_system if os.path.exists(p)), None)

if video_path is None or system_path is None:
    try:
        from google.colab import files
        print("📂 CSV files were not found locally.")
        print("Upload video_telemetry.csv and system_events.csv.")
        uploaded = files.upload()

        # Detect uploaded files by their column structure.
        for name in uploaded:
            temp = pd.read_csv(name, nrows=5)
            if {"timestamp", "eye_gaze_angle", "audio_db"}.issubset(temp.columns):
                video_path = name
            elif {"timestamp", "tab_switches", "is_cheating"}.issubset(temp.columns):
                system_path = name
    except ImportError:
        pass

if video_path is None or system_path is None:
    raise FileNotFoundError(
        "Could not find both datasets. Place video_telemetry.csv and "
        "system_events.csv in the notebook working directory."
    )

print("Video dataset:", video_path)
print("System events dataset:", system_path)

# ==========================================
# 1B. LOAD AND PARSE DATA
# ==========================================
video_df = pd.read_csv(video_path)
sys_df = pd.read_csv(system_path)

video_df["timestamp"] = pd.to_datetime(video_df["timestamp"])
sys_df["timestamp"] = pd.to_datetime(sys_df["timestamp"])

video_df = video_df.sort_values("timestamp").reset_index(drop=True)
sys_df = sys_df.sort_values("timestamp").reset_index(drop=True)

print("Video shape:", video_df.shape)
print("System-events shape:", sys_df.shape)
print("\nVideo columns:", video_df.columns.tolist())
print("System columns:", sys_df.columns.tolist())

In [ ]:
# ==========================================
# 1C. ASYNCHRONOUS MERGE
# ==========================================
merged_df = pd.merge_asof(
    video_df,
    sys_df,
    on="timestamp",
    direction="backward"
)

print("✅ Asynchronous merge complete.")
print("Merged shape:", merged_df.shape)
display(merged_df.head())

## Phase 2 — Missing Data & Feature Engineering

The company requires the exam timeline to be preserved when video frames are missing. Therefore, missing eye-gaze values are forward-filled and missing audio values are interpolated. Missing event/label values created by the asynchronous merge are treated as no event (`0`).

A 10-second rolling window is then used to reduce sensitivity to one-second movements or short background sounds.

In [ ]:
# ==========================================
# 2A. INSPECT MISSING VALUES BEFORE CLEANING
# ==========================================
print("Missing values before imputation:")
display(merged_df.isna().sum())

# ==========================================
# 2B. IMPUTE MISSING SIGNALS
# ==========================================
merged_df["eye_gaze_angle"] = merged_df["eye_gaze_angle"].ffill()
merged_df["eye_gaze_angle"] = merged_df["eye_gaze_angle"].bfill()

merged_df["audio_db"] = merged_df["audio_db"].interpolate(
    method="linear", limit_direction="both"
)

merged_df["tab_switches"] = merged_df["tab_switches"].fillna(0)
merged_df["is_cheating"] = merged_df["is_cheating"].fillna(0)

print("\nMissing values after imputation:")
display(merged_df.isna().sum())

In [ ]:
# ==========================================
# 2C. 10-SECOND ROLLING FEATURES
# ==========================================
merged_df["gaze_rolling_10s"] = (
    merged_df["eye_gaze_angle"].rolling(window=10, min_periods=10).mean()
)

merged_df["audio_rolling_10s"] = (
    merged_df["audio_db"].rolling(window=10, min_periods=10).max()
)

# Remove only rows that cannot have a complete 10-second rolling window.
merged_df = merged_df.dropna().reset_index(drop=True)

print("✅ Feature engineering complete.")
print("Current dataframe shape:", merged_df.shape)

display(
    merged_df[
        [
            "timestamp",
            "eye_gaze_angle",
            "audio_db",
            "tab_switches",
            "gaze_rolling_10s",
            "audio_rolling_10s",
            "is_cheating",
        ]
    ].head(12)
)

In [ ]:
# ==========================================
# 2D. TARGET DISTRIBUTION
# ==========================================
print("Target distribution:")
display(merged_df["is_cheating"].value_counts().rename(index={0: "Not Cheating", 1: "Cheating"}))

print("\nTarget percentage:")
display(
    (merged_df["is_cheating"].value_counts(normalize=True) * 100)
    .rename(index={0: "Not Cheating", 1: "Cheating"})
    .round(2)
)

## Phase 3 — Class-Balanced Model Training

The company requires class imbalance to be handled. A Random Forest with `class_weight="balanced"` is used so the model does not simply favor the majority class.

In [ ]:
# ==========================================
# 3A. FEATURES AND TARGET
# ==========================================
features = [
    "eye_gaze_angle",
    "audio_db",
    "tab_switches",
    "gaze_rolling_10s",
    "audio_rolling_10s",
]

X = merged_df[features]
y = merged_df["is_cheating"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training class distribution:")
display(y_train.value_counts())

In [ ]:
# ==========================================
# 3B. TRAIN CLASS-BALANCED RANDOM FOREST
# ==========================================
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

print("✅ Model training complete.")

In [ ]:
# ==========================================
# 3C. FEATURE IMPORTANCE
# ==========================================
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

display(importance.to_frame("importance"))

importance.sort_values().plot(kind="barh", figsize=(9, 5))
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Phase 4 — Ethical AI Tuning

The company explicitly requires probability-based decisions rather than standard `.predict()`. The strict decision boundary is **0.90**, meaning a record is flagged only when the model estimates at least 90% probability of cheating.

In [ ]:
# ==========================================
# 4A. RAW PROBABILITIES
# ==========================================
y_proba = model.predict_proba(X_test)
cheating_probabilities = y_proba[:, 1]

# ==========================================
# 4B. CUSTOM DECISION THRESHOLDS
# ==========================================
y_pred_default = (cheating_probabilities >= 0.50).astype(int)
y_pred_strict = (cheating_probabilities >= 0.90).astype(int)

def evaluate_threshold(name, y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp,
        "True Negatives": tn,
    }

comparison = pd.DataFrame([
    evaluate_threshold("Default", y_test, cheating_probabilities, 0.50),
    evaluate_threshold("Strict", y_test, cheating_probabilities, 0.90),
])

display(comparison.set_index("Threshold"))

In [ ]:
# ==========================================
# 4C. CONFUSION MATRICES AND REPORTS
# ==========================================
print("🚨 DEFAULT THRESHOLD (0.50)")
print(confusion_matrix(y_test, y_pred_default))
print(classification_report(y_test, y_pred_default, zero_division=0))

print("\n🛡️ STRICT THRESHOLD (0.90)")
print(confusion_matrix(y_test, y_pred_strict))
print(classification_report(y_test, y_pred_strict, zero_division=0))

In [ ]:
# ==========================================
# 4D. PRECISION-RECALL CURVE
# ==========================================
precision, recall, thresholds = precision_recall_curve(
    y_test,
    cheating_probabilities
)

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")
plt.axvline(
    0.90,
    linestyle="--",
    label="Strict threshold = 0.90"
)
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision and Recall vs Decision Threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show the nearest evaluated threshold to 0.90.
if len(thresholds):
    idx = np.argmin(np.abs(thresholds - 0.90))
    print(f"Nearest evaluated threshold to 0.90: {thresholds[idx]:.4f}")
    print(f"Precision there: {precision[idx]:.4f}")
    print(f"Recall there: {recall[idx]:.4f}")

## Phase 5 — Final Executive Recommendation

### Recommendation

The production decision should use the **strict 90% probability threshold** because the business requirement prioritizes protecting innocent students from false accusations.

In this dataset, the strict threshold is evaluated directly against the test set. The table above shows how moving from 0.50 to 0.90 changes false positives, precision, recall, and the overall trade-off.

The 10-second rolling features reduce sensitivity to isolated one-second movements or short audio spikes by incorporating a short history of telemetry rather than reacting to a single observation.

### Deployment Status

**GO — with the strict 90% threshold and continued monitoring.**

This recommendation is conditional on maintaining the stated precision-first policy and validating the model on additional real-world exam sessions before high-stakes deployment.

In [ ]:
# ==========================================
# FINAL EXECUTIVE METRICS
# ==========================================
strict_metrics = evaluate_threshold(
    "Strict 90%",
    y_test,
    cheating_probabilities,
    0.90
)

print("FINAL EXECUTIVE METRICS")
for key, value in strict_metrics.items():
    if key != "Threshold":
        print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

print("\n✅ Executive notebook pipeline completed successfully.")